In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

✨🍰✨ Everything looks OK!


In [ ]:
!conda install -c conda-forge openmm openff-toolkit openff-interchange lxml rdkit pdbfixer biopython scikit-learn plotly py3Dmol pytest vina meeko -y

Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / - done

# All requested packages already installed.



In [ ]:
!pip install "numpy>=2.0,<2.3" "pandas>=2.2,<3" "scipy>=1.12,<2" streamlit torch torchvision wheel openmmforcefields gemmi meeko psutil stmol py3Dmol ipython_genutils

In [ ]:
!pip install --upgrade meeko

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 12.9 MB/s  0:00:00
  Attempting uninstall: meeko
    Found existing installation: meeko 0.5.0
    Uninstalling meeko-0.5.0:
      Successfully uninstalled meeko-0.5.0


In [ ]:
!unzip AMR-UNBIND-v1.3.2.zip
%cd AMR-UNBIND-v1.3.2

unzip:  cannot find or open AMR-UNBIND-v1.3.2.zip, AMR-UNBIND-v1.3.2.zip.zip or AMR-UNBIND-v1.3.2.zip.ZIP.
[Errno 2] No such file or directory: 'AMR-UNBIND-v1.3.2'
/content/AMR-UNBIND-v1.3.2


In [ ]:
import os
import glob

# 1. Locate app.py
matches = glob.glob('/content/**/app.py', recursive=True)
if not matches:
    print("❌ app.py not found. Re-run your unzip cell: !unzip *.zip")
else:
    target_dir = os.path.dirname(matches[0])
    os.chdir(target_dir)

    # 2. Download Cloudflare Tunnel binary if missing
    if not os.path.exists('/usr/local/bin/cloudflared'):
        print("⚡ Installing Cloudflare Tunnel...")
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
        !chmod +x /usr/local/bin/cloudflared

    # 3. Launch Streamlit and open Cloudflare Tunnel
    print("🚀 Starting Streamlit...")
    !python -m streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false & cloudflared tunnel --url http://localhost:8501


🚀 Starting Streamlit...
2026-09-15T07:27:08Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-15T07:27:08Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-09-15 07:27:09.411 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.30.75:8501

2026-09-15T07:27:11Z INF +--------------------------------------

In [ ]:
import os
import glob

matches = glob.glob('/content/**/app.py', recursive=True)
if matches:
    os.chdir(os.path.dirname(matches[0]))
    !python -m streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false & cloudflared tunnel --url http://localhost:8501
else:
    print("❌ app.py not found. Re-run your unzip cell (!unzip *.zip) first.")


/bin/bash: line 1: cloudflared: command not found


2026-09-15 14:58:38.954 Port 8501 is not available


In [ ]:
import os
import glob
import subprocess
import time
import re

# 1. Locate app.py directory
matches = glob.glob('/content/**/app.py', recursive=True)
if matches:
    os.chdir(os.path.dirname(matches[0]))

# 2. Kill existing processes
!pkill -f streamlit
!pkill -f cloudflared

# Ensure Cloudflare Tunnel binary is present
if not os.path.exists('/usr/local/bin/cloudflared'):
    print("⚡ Installing Cloudflare Tunnel...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    print("Cloudflare Tunnel installed.")
else:
    print("Cloudflare Tunnel binary already exists.")

# 3. Launch processes writing to log files
st_log = open("streamlit.log", "w")
cf_log = open("cloudflare.log", "w")

subprocess.Popen(
    ["python", "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=st_log, stderr=subprocess.STDOUT
)
subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=cf_log, stderr=subprocess.STDOUT
)

print("⏳ Initializing Streamlit and Cloudflare Tunnel...")
url_found = False

# 4. Keep cell execution active to prevent Colab process termination
try:
    while True:
        time.sleep(2)
        if not url_found and os.path.exists("cloudflare.log"):
            with open("cloudflare.log", "r") as f:
                content = f.read()
                match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
                if match:
                    print(f"\n✅ APP IS LIVE! Access your dashboard here:\n{match.group(0)}\n")
                    print("📌 Keep this cell running while using the app.")
                    url_found = True
except KeyboardInterrupt:
    print("Stopping tunnel...")

Cloudflare Tunnel binary already exists.
⏳ Initializing Streamlit and Cloudflare Tunnel...

✅ APP IS LIVE! Access your dashboard here:
https://durham-herbal-cookie-prescription.trycloudflare.com

📌 Keep this cell running while using the app.


In [ ]:
import os
import subprocess
import time
import re
import requests

os.chdir('/content/AMR-UNBIND-v1.2.0')

# Kill any previous processes
!pkill -f streamlit
!pkill -f cloudflared

# Fresh logs each run
st_log = open("streamlit.log", "w")
cf_log = open("cloudflare.log", "w")

# Relaunch Streamlit
subprocess.Popen(
    ["python", "-m", "streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=st_log, stderr=subprocess.STDOUT
)

# Ensure Cloudflare Tunnel binary is present
if not os.path.exists('/usr/local/bin/cloudflared'):
    print("⚡ Re-installing Cloudflare Tunnel as it was not found...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    print("Cloudflare Tunnel re-installed.")
else:
    print("Cloudflare Tunnel binary already exists.")

# Relaunch Cloudflare tunnel with no auto-update
subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8501"],
    stdout=cf_log, stderr=subprocess.STDOUT
)

print("⏳ Relaunching server...")
time.sleep(5)

# Extract tunnel URL
url_found = False
tunnel_url = None

try:
    while True:
        time.sleep(2)
        if not url_found and os.path.exists("cloudflare.log"):
            with open("cloudflare.log", "r") as f:
                content = f.read()[-5000:]  # rotate: only read last 5k chars
                match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
                if match:
                    tunnel_url = match.group(0)
                    print(f"\n✅ APP IS LIVE! Access your dashboard here:\n{tunnel_url}\n")
                    print("📌 Keep this cell running while using the app.")
                    url_found = True

        # Keep-alive ping every 60s if tunnel URL is known
        if tunnel_url:
            try:
                requests.get(tunnel_url, timeout=5)
                print(f"🔄 Keep-alive ping sent.")
            except Exception as e:
                print(f"⚠️ Ping failed: {e}")

except KeyboardInterrupt:
    print("Stopping tunnel...")

# Fallback: if Cloudflare drops, you can try ngrok/localtunnel
# !pip install pyngrok
# from pyngrok import ngrok
# ngrok.connect(8501)

⚡ Re-installing Cloudflare Tunnel as it was not found...
Cloudflare Tunnel re-installed.
⏳ Relaunching server...

✅ APP IS LIVE! Access your dashboard here:
https://martin-possible-nominations-belly.trycloudflare.com

📌 Keep this cell running while using the app.
⚠️ Ping failed: HTTPSConnectionPool(host='martin-possible-nominations-belly.trycloudflare.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='martin-possible-nominations-belly.trycloudflare.com', port=443): Failed to resolve 'martin-possible-nominations-belly.trycloudflare.com' ([Errno -2] Name or service not known)"))
⚠️ Ping failed: HTTPSConnectionPool(host='martin-possible-nominations-belly.trycloudflare.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='martin-possible-nominations-belly.trycloudflare.com', port=443): Failed to resolve 'martin-possible-nominations-belly.trycloudflare.com' ([Errno -2] Name or service not 

In [ ]:
!tail -n 30 streamlit.log



2026-09-15 14:58:46.049 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.108.97.153:8501



In [ ]:
import os
import glob
import re
import subprocess
import time

# 1. Locate and set project working directory
matches = glob.glob('/content/**/app.py', recursive=True)
if matches:
    root_dir = os.path.dirname(matches[0])
    os.chdir(root_dir)
    print(f"📁 Project root set to: {root_dir}")

# 2. Terminate any stale background instances
!pkill -f streamlit
!pkill -f cloudflared

# 3. Apply Source Code Patches (Fixes NaN Errors & 15% Parameterization Freezes)
py_files = glob.glob(f"{os.getcwd()}/**/*.py", recursive=True)

for filepath in set(py_files):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    modified = False

    # Fix 1: Resolve 15% OpenFF freeze by swapping slow AM1-BCC charges for Gasteiger
    if 'am1bcc' in content:
        content = content.replace("'am1bcc'", "'gasteiger'").replace('"am1bcc"', '"gasteiger"')
        modified = True

    # Fix 2: Prevent NaN crash by forcing Energy Minimization BEFORE velocity assignment
    if 'setVelocitiesToTemperature' in content and 'minimizeEnergy' in content:
        pattern = r'(setPositions\([^)]+\))\s*\n\s*(setVelocitiesToTemperature\([^)]+\))\s*\n\s*(minimizeEnergy\([^)]*\))'
        if re.search(pattern, content):
            content = re.sub(pattern, r'\1\n    min_sim.minimizeEnergy(maxIterations=3000)\n    \2', content)
            modified = True
        elif 'minimizeEnergy()' in content:
            content = content.replace('minimizeEnergy()', 'minimizeEnergy(maxIterations=3000)')
            modified = True

    if modified:
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f"✅ Applied code patch to: {os.path.basename(filepath)}")

# 4. Sanitize PDB Topology (Fixes Meeko "No template matched for residue A:151" error)
pdb_files = glob.glob('/content/**/*.pdb', recursive=True)
for pdb_path in set(pdb_files):
    if os.path.exists(pdb_path):
        with open(pdb_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        clean_lines = [l for l in lines if not l.startswith('CONECT')]
        with open(pdb_path, 'w', encoding='utf-8') as f:
            f.writelines(clean_lines)

print("✅ Sanitized PDB topology files (stripped CONECT records).")

# 5. Persistent Foreground Subprocess Launcher (Prevents Colab Process Termination)
st_log = open("streamlit.log", "w")
cf_log = open("cloudflare.log", "w")

subprocess.Popen(
    ["python", "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=st_log, stderr=subprocess.STDOUT
)
subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=cf_log, stderr=subprocess.STDOUT
)

print("⏳ Initializing server and Cloudflare tunnel...")
url_found = False

try:
    while True:
        time.sleep(2)
        if not url_found and os.path.exists("cloudflare.log"):
            with open("cloudflare.log", "r") as f:
                log_txt = f.read()
                match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_txt)
                if match:
                    print(f"\n✅ PIPELINE IS READY & ONLINE!")
                    print(f"🔗 Access App: {match.group(0)}\n")
                    print("📌 Leave this Colab cell running while testing the app.")
                    url_found = True
except KeyboardInterrupt:
    print("\n🛑 Server stopped manually.")

📁 Project root set to: /content/AMR-UNBIND-v1.2.0
✅ Applied code patch to: test_bug_fixes.py
✅ Sanitized PDB topology files (stripped CONECT records).
⏳ Initializing server and Cloudflare tunnel...

✅ PIPELINE IS READY & ONLINE!
🔗 Access App: https://relation-try-edited-retain.trycloudflare.com

📌 Leave this Colab cell running while testing the app.


In [ ]:
import pdbfixer
import openmm.app as app
from google.colab import files

# 1. Load and clean 3PTB
fixer = pdbfixer.PDBFixer(pdbid="3PTB")
fixer.missingResidues = {}  # Wipe phantom residues (1-15) from SEQRES header
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.4)

# 2. Save the structure
output_filename = "3PTB_clean.pdb"
with open(output_filename, "w") as f:
    app.PDBFile.writeFile(fixer.topology, fixer.positions, f)

# 3. Download directly to your computer's Downloads folder
files.download(output_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
import glob, shutil
from google.colab import files

# Find the latest run folder
run_dirs = sorted(glob.glob("amr_workspace/runs/*3ptb*"))
latest_run = run_dirs[-1]

# Zip the run directory and download
shutil.make_archive("smd_run_results", "zip", latest_run)
files.download("smd_run_results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>